In [1]:
import numpy as np
import pandas as pd
import gc
from tqdm import tqdm
from os.path import join
import os
import sys
import yaml
from importlib import reload

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


# Tokenizer [CPU]

In [3]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

In [11]:
bos_id = 0
eos_id = 1
pad_id = 2

### Train tokenizer

In [55]:
tokenizer = Tokenizer(BPE())
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
tokenizer.decoder = ByteLevelDecoder()

In [56]:
special_tokens = ["<bos>", "<eos>", "<pad>"]

trainer = BpeTrainer(
    vocab_size=32_000, 
    special_tokens=special_tokens,
    initial_alphabet=ByteLevel.alphabet()
)

In [57]:
dataset = pd.read_csv(
    "/content/drive/MyDrive/machine_translation/datasets/en-es/EN-ES.txt", 
    sep='\t', 
    header = None,
    encoding='utf-8',
)[[0,1]].rename(columns = {0: "EN", 1: "ES"})

In [58]:
corpus = dataset['EN'].dropna().astype(str).tolist() + dataset['ES'].dropna().astype(str).tolist()
del dataset
gc.collect()

131

In [59]:
tokenizer.train_from_iterator(corpus, trainer=trainer)
tokenizer.save("/content/drive/MyDrive/machine_translation/datasets/en-es/bpe.json")

In [61]:
encoded = tokenizer.encode("Hola! Let's check out this custom BPE.")
encoded.ids, encoded.tokens, tokenizer.decode(encoded.ids)

([2697, 269, 3, 8404, 1475, 4815, 852, 633, 4956, 545, 1315, 16],
 ['Ho',
  'la',
  '!',
  'ĠLet',
  "'s",
  'Ġcheck',
  'Ġout',
  'Ġthis',
  'Ġcustom',
  'ĠB',
  'PE',
  '.'],
 "Hola! Let's check out this custom BPE.")

### Encode data using trained tokenizer

In [4]:
dataset = pd.read_csv(
    "/content/drive/MyDrive/machine_translation/datasets/en-es/EN-ES.txt", 
    sep='\t', 
    header = None,
    encoding='utf-8',
)[[0,1]].rename(columns = {0: "EN", 1: "ES"})
dataset = dataset.dropna(how="any")

In [5]:
tokenizer = Tokenizer.from_file("/content/drive/MyDrive/machine_translation/datasets/en-es/bpe.json")

In [6]:
def get_lengths(dataset, batch_size=10_000):
    lengths = []

    for i in tqdm(range(0, len(dataset), batch_size), desc="Measuring lengths"):
        batch = dataset[i: i + batch_size]

        encoded = tokenizer.encode_batch_fast(batch)

        lengths.extend([len(output.ids) for output in encoded])

        del encoded
        gc.collect()

    return lengths

In [7]:
lengths_en = np.array(get_lengths(dataset["EN"]), dtype=np.uint16)
lengths_es = np.array(get_lengths(dataset["ES"]), dtype=np.uint16)
lengths = np.concat((lengths_en, lengths_es))

Measuring lengths: 100%|██████████| 570/570 [08:58<00:00,  1.06it/s]


In [8]:
# get length statistics
np.max(lengths), np.percentile(lengths, 99)

(np.uint16(926), np.float64(104.0))

In [12]:
tokenizer = Tokenizer.from_file("/content/drive/MyDrive/machine_translation/datasets/en-es/bpe.json")
max_len = 128

In [13]:
# get length mask
length_mask = (lengths_en <= max_len) & (lengths_es <= (max_len - 2))   # Decoder input also needs to account for BOS + EOS
length_mask.sum()

np.int64(5664938)

In [18]:
def encode_batch_and_dump(tokenizer, dataset, filename, max_len = 128, batch_size = 10_000, decoder=False):
    tokenizer.enable_padding(
        direction="right", 
        pad_id=pad_id,
        pad_token="<pad>", 
        length=(max_len - 2) if decoder else max_len,   # Decoder input also needs to account for BOS + EOS
    )

    n = dataset.shape[0]

    mmap = np.memmap(
        filename,
        dtype=np.uint16,
        mode="w+",
        shape=(n, max_len),
    )

    for start in tqdm(range(0, n, batch_size), desc="Encoding and dumping"):
        end = min(start + batch_size, n)

        batch = dataset[start:end]
        encoded = tokenizer.encode_batch(batch)

        # print([len(output.ids) for output in encoded if len(output.ids) != max_len])

        matrix_slice = np.asarray(
            [output.ids for output in encoded],
            dtype=np.uint16,
        )

        if decoder:
            mmap[start:end, 1:-1] = matrix_slice
        else:
            mmap[start:end] = matrix_slice

        del batch, encoded, matrix_slice
        gc.collect()

    if decoder:
        mmap[:, 0] = bos_id
        mmap[:, -1] = eos_id

    mmap.flush()
    del mmap
    gc.collect()

In [19]:
encode_batch_and_dump(
    tokenizer,
    dataset.loc[length_mask, "EN"], 
    "/content/drive/MyDrive/machine_translation/datasets/en-es/EN.npy", 
    max_len=max_len,
    batch_size = 10_000,
)

Encoding and dumping: 100%|██████████| 567/567 [12:07<00:00,  1.28s/it]


In [20]:
encode_batch_and_dump(
    tokenizer,
    dataset.loc[length_mask, "ES"], 
    "/content/drive/MyDrive/machine_translation/datasets/en-es/ES.npy",
    max_len=max_len,
    batch_size = 10_000,
    decoder=True
)

Encoding and dumping: 100%|██████████| 567/567 [14:24<00:00,  1.53s/it]


In [24]:
a = np.memmap(
    "/content/drive/MyDrive/machine_translation/datasets/en-es/ES.npy",
    mode="r",
    dtype=np.uint16,
    shape=(length_mask.sum(), max_len)
)
a[:5, -2]

memmap([2, 2, 2, 2, 2], dtype=uint16)

# Train Model [TPU]

In [3]:
!pip install torch-xla

In [5]:
# Make sure all the scripts are in place
# Pull from github

# 1. Define repo details
REPO_URL = "https://github.com/arka816/langlocal.git"
REPO_DIR = "/content/langlocal"

!rm -rf {REPO_DIR}

# 2. Clone if the directory doesn't exist (e.g., after a runtime crash)
if not os.path.exists(REPO_DIR):
    print("Runtime disconnected. Re-cloning scripts...")
    !git clone {REPO_URL}
else:
    # If the runtime didn't crash but you updated code locally, pull the latest changes
    print("Runtime active. Pulling latest script changes...")
    !cd {REPO_DIR} && git pull

# 3. CRITICAL: Add the cloned directory to Python's search path so you can import from it
if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

Runtime disconnected. Re-cloning scripts...
Cloning into 'langlocal'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 102 (delta 49), reused 80 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (102/102), 48.42 KiB | 6.92 MiB/s, done.
Resolving deltas: 100% (49/49), done.


In [6]:
# import machine_translation.data_pipeline.collator as collator
# reload(collator)

# import machine_translation.transformer.train as train
# reload(train)

from machine_translation.transformer.train import train_tpu_single_core

In [7]:
config_path = join(REPO_DIR, "machine_translation", "transformer", "config.yaml")

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

config

{'data': {'VOCAB_SIZE': 32000, 'SENTENCE_SIZE': 128, 'DATA_SIZE': 5667063},
 'training': {'NUM_EPOCHS': 10, 'BATCH_SIZE': 128},
 'model': {'EMBEDDING_DIM': 512,
  'FFN_DIM': 2048,
  'HEADS': 8,
  'NUM_LAYERS': 6,
  'DROPOUT': 0.1,
  'SHARE_EMBEDDINGS': True}}

In [11]:
src_drive_file = "/content/drive/MyDrive/machine_translation/datasets/en-es/EN.npy"
tgt_drive_file = "/content/drive/MyDrive/machine_translation/datasets/en-es/ES.npy"

src_file = "/content/EN.npy"
tgt_file = "/content/ES.npy"

!cp {src_drive_file} /content/
!cp {tgt_drive_file} /content/

In [ ]:
data_size = config['data']['DATA_SIZE']
sentence_size = config['data']['SENTENCE_SIZE']
file_dims = (data_size, sentence_size)

model_configs = {
    "src_vocab":    config['data']['VOCAB_SIZE'],
    "tgt_vocab":    config['data']['VOCAB_SIZE'],
    "N":            config['model']['NUM_LAYERS'],
    "d_model":      config['model']['EMBEDDING_DIM'], 
    "d_ff":         config['model'].get("FFN_DIM", 4 * config['model']['EMBEDDING_DIM']), 
    "heads":        config['model']['HEADS'],
    "dropout":      config['model']['DROPOUT'], 
    "share_embedding":  config['model']['SHARE_EMBEDDINGS']
}

epochs = config['training']['NUM_EPOCHS']
batch_size = config['training']['BATCH_SIZE']
loader_workers = 4 #config['training']['LOADER_WORKERS']

checkpoint_filepath =  "/content/drive/MyDrive/machine_translation/en-es-machine-translator.pt"

dynamic_padding = config['training']['DYNAMIC_PADDING']

In [ ]:
print("Kicking off training...")
train_tpu_single_core(
    src_file,
    tgt_file,
    file_dims,
    model_configs,
    bos_id=bos_id,
    eos_id=eos_id,
    pad_id=pad_id,
    batch_size=batch_size,
    epochs=epochs,
    loader_workers=loader_workers,
    checkpoint_filepath=checkpoint_filepath,
    dynamic_padding=dynamic_padding
)

Kicking off training...
Using XLA device: xla:0
Number of parameters:  60524544


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:841: UserWarning: aten::std: an autograd kernel was not registered to the Autograd key(s) but we are trying to backprop through it. This may lead to silently incorrect behavior. This behavior is deprecated and will be removed in a future version of PyTorch. If your operator is differentiable, please ensure you have registered an autograd kernel to the correct Autograd key (e.g. DispatchKey::Autograd, DispatchKey::CompositeImplicitAutograd). If your operator is not differentiable, or to squash this warning and use the previous behavior, please register torch::CppFunction::makeFallthrough() to DispatchKey::Autograd. (Triggered internally at /pytorch/torch/csrc/autograd/autograd_not_implemented_fallback.cpp:62.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


RuntimeError: Error loading program 'SyncTensorsGraph.19692': Attempting to allocate 274.86M. That was not possible. There are 241.87M free.; (0x0x0_HBM0)